# BioPAP Tutorial 2: Create a Pipeline

This tutorial shows how to build and run a preprocessing pipeline with BioPAP's `Pipeline` class. A pipeline applies processing steps in sequence, passing each step's output to the next.

Use this tutorial to:

- Create a pipeline for your data and add processing steps in the order you want them to run.
- Overwrite module configurations from the configuration file.
- Run the complete pipeline and inspect its steps.
- Save the pipeline configuration for reuse, sharing, or batch processing
- Re-create a pipeline from configuration file

### Configuration File

If you don't have a configuration file. Please look at the example configuration file: `./data/pipeline.yaml.example`. You can find example settings of the modules in the module's documentation ([API documentation](../api/)).

If you don't know the correct configurations for the modules, please have a look at [tutorial 1](./1_explore_modules.ipynb).

In [ ]:
# Install latest version from local files (only necessary after pulling updated version from Git).
# Restart Jupyter kernel when install has finished.
%pip install -e ..

In [ ]:
# Some Windows installations require explicitly installing mne (required only once; restart Jupyter kernel after installing).
%pip install mne mne_bids

In [ ]:
from biopap.pipeline import PipelineStep, Pipeline

from biopap.modules.load_data import LoadData
from biopap.modules.anonymize import Anonymize
from biopap.modules.rereference import Rereference
from biopap.modules.set_channel_type_and_montage import SetChannelTypeAndMontage
from biopap.modules.eog_channels import EOGChannels
from biopap.modules.blink_detection import BlinkDetection
from biopap.modules.iir_filter import IIRFilter
from biopap.modules.downsample import Downsample
from biopap.modules.segmentation import Segmentation
from biopap.modules.artifact_reject import ArtifactReject
from biopap.modules.baseline_correction import BaselineCorrection
from biopap.modules.save_output import SaveOutput
from biopap.modules.bids import BIDSConversion
from biopap.modules.plot import Plot

In [ ]:
import mne
from pathlib import Path
from pprint import pprint

In [ ]:
# To avoid issues with local settings and GitHub updates, create a small notebook
# called 'my_config.ipynb' to hold variables that are likely to change locally. 
# In the same folder as this file, there is an example notebook called
# 'example.my_config.ipynb' that can serve as a template.

config_notebook = './data/my_config.ipynb'

if Path(config_notebook).exists():
    %run {config_notebook}
else:
    data_file = './data/example.bdf'
    config_file = './data/pipeline.yaml'

if not Path(config_file).exists():
    raise FileNotFoundError(f"Config file {config_file} does not exist.")

if not Path(data_file).exists():
    raise FileNotFoundError(f"Data file {data_file} does not exist.")

### Creating a Pipeline Instance

Create a new pipeline by instantiating the `Pipeline` class with your data and configuration file:


In [ ]:
# Read the data file and load it into memory.
load = LoadData(file_path=data_file)

In [ ]:
# Create a new pipeline, loading the raw data and the configuration file
pipe = Pipeline(data=load.raw, config=config_file, verbose='ERROR')

In [ ]:
# Inspect the loaded configuration
print("Pipeline Configuration:")
pprint(pipe.config)

### Adding Steps to the Pipeline

A pipeline is made up of processing steps executed in order. Each step's output becomes the next step's input.

**Key points:**
- Do NOT specify input data when adding steps—the pipeline automatically chains outputs
- Settings are read from the configuration file by default
- Steps are not executed until you call `pipe.run()`
- You can reset the pipeline and rebuild it if needed

#### Example 1: Add basic steps with default configuration

In [ ]:
# Add processing steps in order
pipe.add_step(Anonymize)
pipe.add_step(Downsample)
pipe.add_step(IIRFilter)
pipe.add_step(Plot)

# Print pipeline overview (steps and their configuration)
print("Pipeline Overview:")
print(*pipe.steps, sep='\n')

#### Example 2: Using configuration labels for same module with different settings

When you need to run the same module multiple times with different configurations, use a `cfg_label` to distinguish them in the config file:


In [ ]:
# Example: Use Downsample twice with different config labels
pipe.add_step(Downsample, cfg_label='Downsample_500Hz')
pipe.add_step(Downsample, cfg_label='Downsample_1000Hz')  # Different settings in config file

print("Pipeline with multiple Downsample steps:")
print(*pipe.steps, sep='\n')

#### Example 3: Override configuration parameters on-the-fly

You can override configuration file settings by passing a `cfg` dictionary:


In [ ]:
# Example: Override configuration with custom parameters
pipe.add_step(IIRFilter, cfg={'h_freq': 50})  # Use 50 Hz instead of config file value

print("IIRFilter with overridden frequency:")
print(*pipe.steps, sep='\n')

#### Resetting the Pipeline

To clear the pipeline and start over with a different configuration, use `reset()`:


In [ ]:
# Clear the pipeline to start fresh
pipe.reset()
print("Pipeline has been reset")

### Complete Pipeline Example

Build a complete preprocessing pipeline with the recommended processing order:


In [ ]:
# Add all processing steps in the recommended order
pipe.add_step(Anonymize)
pipe.add_step(Rereference)
pipe.add_step(SetChannelTypeAndMontage)
pipe.add_step(EOGChannels)
pipe.add_step(IIRFilter)
pipe.add_step(Downsample)
pipe.add_step(Segmentation)
pipe.add_step(BaselineCorrection)
pipe.add_step(SaveOutput)

### Inspect the  pipeline

In [ ]:
# Display the complete pipeline
pipe.inspect()

### Running the Pipeline

Once you've added all the desired steps, execute the entire pipeline with a single command:


In [ ]:
# Run the entire pipeline sequentially
pipe.run()

print("Pipeline execution complete!")

### Saving the Pipeline

When the pipeline ran without errors, you can save the pipeline with:


In [ ]:
pipe.to_config('./config.yaml')

### Saving changed configuration

Should you add a step that includes ad-hoc configuration to your pipeline, the extra or updated configuration settings are not part of the configuration file you created the pipeline with. To make sure you save the most up to date version of the configuration, call the pipeline's 'to_config()` method.

For instance, in the example configuration, the ArtifactReject-module has a value of 100 for the variable 'interval_ms', but when adding the module to the pipeline, the value could be updated to 200:

In [ ]:
pipe.reset()

pipe.add_step(Anonymize)
pipe.add_step(Rereference)
pipe.add_step(SetChannelTypeAndMontage)
pipe.add_step(EOGChannels)
pipe.add_step(IIRFilter)
pipe.add_step(Downsample, cfg={'sfreq': 400})
pipe.add_step(BlinkDetection)
pipe.add_step(Segmentation)
pipe.add_step(ArtifactReject, cfg={'interval_ms':200})
pipe.add_step(BaselineCorrection)
pipe.add_step(SaveOutput)

pipe.to_config('./updated_config.yaml')

Please note that in the config file, the steps will likely end up in the file sorted by step name, rather than by step order. This is to be expected, as the Yaml-format does not guarantee the order of its elements. However, when the pipeline reads back the config file, it will sort the steps by step order, recreating the intended order or processing.

---

# Re-creating pipeline from config file

By loading a saved config file into a new `Pipeline`-instance, you automatically create the complete pipeline with all the specified settings.

In [ ]:
cfg_file = './updated_config.yaml'

new_pipe = Pipeline.from_config(
    config=cfg_file,
    input_file=data_file,
    subject_id='ABC123'
)

new_pipe.inspect()

## A word on subject_id

The subject ID can be used to differentiate between outputs for different users when running the pipeline in batches. When you use the string `{subject_id}` in the value of any string-type config variable, it will be automatically replaced by the value of subject_id. So, for instance, by defining a pipeline-step thus:

```python
SaveOutput(
    raw=baseline.processed,
    export_formats='fif',
    output_file='./partially_processed_{subject_id}.fif',
    overwrite=True)
```
running the pipeline with `subject_id="A123"` will result in an outputfile `partially_processed_A123.fif`.

If you do not specify a subject_id when creating the pipeline, the program will use the input file's name without the suffix. It will first try to read this from the 'filenames' attribute from the file's metadata, and if that is empty, use the actual file's path.